# Adaptive Trust Gate — Multi-Seed Full Pipeline (Colab)

Clones the `adaptive-trust-gate` repo, installs dependencies, fetches the raw MovieLens `ml-latest-small` dataset (not committed to git — it's gitignored under `data/raw/`), and runs `scripts/10_multiseed_full.py`: the 7-model comparison (CF-SVD++, content-based, static/learned/bandit/GA/sequential hybrid gates) across 5 seeds.

The repo is **private**, so the clone cell asks for a GitHub personal access token (`repo` scope). It's entered via `getpass` so it's only used locally in this Colab runtime — never stored in the notebook or shown in output.

In [ ]:
import getpass

token = getpass.getpass("GitHub personal access token (repo scope): ")
repo_url = f"https://{token}@github.com/Ar555Rathod/adaptive-trust-gate.git"
!git clone -q {repo_url}
%cd adaptive-trust-gate
del token, repo_url

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Raw dataset is gitignored (not committed) -- fetch the official small MovieLens release directly.
!mkdir -p data/raw
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O data/raw/ml-latest-small.zip
!unzip -oq data/raw/ml-latest-small.zip -d data/raw
!ls data/raw/ml-latest-small

In [ ]:
!python scripts/10_multiseed_full.py

## Inspect the results

In [ ]:
import json
import pandas as pd

with open("results/metrics/multiseed_full_comparison.json") as f:
    out = json.load(f)

rows = []
for name, agg in out["aggregate"].items():
    row = {"model": name}
    for seg in ("overall", "cold", "warm", "power"):
        if seg in agg:
            row[f"{seg}_rmse_mean"] = agg[seg]["rmse_mean"]
            row[f"{seg}_rmse_std"] = agg[seg]["rmse_std"]
    rows.append(row)

pd.DataFrame(rows).set_index("model")